# Exploration de l’inflation au Maroc

Projet individuel de première année à l’INSEA. Ce notebook décrit l’évolution de l’IPC mensuel national et prépare une analyse du pouvoir d’achat.

## 1. Présentation des données

Source de référence : [indicateurs IPC du HCP](https://www.hcp.ma/Indicateurs-Indices-des-prix-a-la-consommation-IPC_r351.html).

- Fichier et lien exact de la source : à renseigner après téléchargement.
- Période et année de base de l’IPC : à renseigner à partir du fichier.
- Champ attendu : indice général du Maroc, une observation par mois, avec une base constante.

Déposer le fichier source dans `data/raw/`. Si le tableau est en PDF, si ses mois figurent en colonnes ou si plusieurs séries sont mélangées, préparer une table mensuelle à deux colonnes dans `data/processed/` et conserver l’original. L’import ci-dessous accepte les fichiers CSV et Excel `.xlsx`.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

# Le notebook peut démarrer depuis la racine ou depuis le dossier notebooks.
racine = Path.cwd()
if racine.name == "notebooks":
    racine = racine.parent

# Adapter ces paramètres après avoir ouvert le fichier officiel.
dossier = racine / "data" / "raw"  # Remplacer "raw" par "processed" si nécessaire.
nom_fichier = ""  # Préciser le nom avec son extension.
colonne_date = "date"
colonne_ipc = "ipc"
format_date = "%Y-%m"  # Pour des dates en texte. Adapter au fichier.
ligne_entete = 0  # 0 désigne la première ligne.
separateur_csv = ";"
encodage_csv = "utf-8-sig"
feuille_excel = 0  # 0 désigne le premier onglet.

donnees_brutes = None

if not nom_fichier:
    print("Renseigner nom_fichier après avoir ajouté les données du HCP.")
else:
    chemin = dossier / nom_fichier
    if not chemin.is_file():
        print(f"Fichier introuvable : {chemin}")
    elif chemin.suffix.lower() == ".csv":
        donnees_brutes = pd.read_csv(
            chemin, sep=separateur_csv, encoding=encodage_csv, header=ligne_entete
        )
    elif chemin.suffix.lower() == ".xlsx":
        donnees_brutes = pd.read_excel(
            chemin, sheet_name=feuille_excel, header=ligne_entete
        )
    else:
        print("Utiliser un fichier CSV ou Excel au format .xlsx.")

if donnees_brutes is not None:
    print("Colonnes disponibles :", donnees_brutes.columns.tolist())
    display(donnees_brutes.head())

## 2. Nettoyage et contrôles

Convertir les dates en mois et l’indice en nombre. Corriger les lignes invalides et les mois en double dans une copie préparée à partir de la source. Vérifier notamment qu’un doublon ne correspond pas à un autre poste de dépense ou à une autre ville. Les mois manquants restent sans valeur.

In [ ]:
donnees = None

if donnees_brutes is not None:
    colonnes = [colonne_date, colonne_ipc]
    if not all(colonne in donnees_brutes.columns for colonne in colonnes):
        print("Adapter les noms des colonnes dans la cellule d’import.")
    elif donnees_brutes.empty:
        print("Le tableau ne contient aucune ligne de données.")
    else:
        dates = donnees_brutes[colonne_date]
        if not pd.api.types.is_datetime64_any_dtype(dates):
            dates = pd.to_datetime(dates, format=format_date, errors="coerce")

        # Accepter la virgule décimale dans les valeurs de l’indice.
        valeurs = donnees_brutes[colonne_ipc].astype("string").str.strip()
        ipc = pd.to_numeric(valeurs.str.replace(",", ".", regex=False), errors="coerce")
        table = pd.DataFrame({"mois": dates.dt.to_period("M"), "ipc": ipc})
        invalides = table["mois"].isna() | ~table["ipc"].gt(0).fillna(False)
        invalides = invalides | table["ipc"].eq(float("inf")).fillna(False)
        doublons = table["mois"].notna() & table["mois"].duplicated(keep=False)

        print("Lignes invalides :", invalides.sum())
        print("Lignes associées à un mois en double :", doublons.sum())
        if invalides.any() or doublons.any():
            display(donnees_brutes.loc[invalides | doublons].head(10))
        else:
            table = table.set_index("mois").sort_index()
            mois_complets = pd.period_range(table.index.min(), table.index.max(), freq="M")
            donnees = table.reindex(mois_complets)
            donnees.index.name = "mois"
            print("Mois manquants :", donnees["ipc"].isna().sum())
            display(donnees.head())

## 3. Analyse descriptive

La variation mensuelle compare l’IPC au mois précédent. Le glissement annuel le compare au même mois de l’année précédente. Dans les deux cas, le taux en pourcentage se calcule par `100 × (IPC courant / IPC de référence − 1)`. Un taux reste manquant lorsque l’un des deux indices nécessaires est absent.

Les statistiques descriptives donnent notamment le nombre d’observations, la moyenne, la médiane et les valeurs extrêmes. Un glissement annuel mensuel se distingue du taux d’inflation moyen d’une année civile.

In [ ]:
if donnees is not None:
    ipc = donnees["ipc"]
    # Le calendrier mensuel complet garantit le bon mois de comparaison.
    donnees["variation_mensuelle_pct"] = 100 * (ipc / ipc.shift(1) - 1)
    donnees["glissement_annuel_pct"] = 100 * (ipc / ipc.shift(12) - 1)
    display(donnees.describe().round(2))

## 4. Visualisation

Tracer d’abord le niveau de l’IPC, puis ses variations en pourcentage. Repérer les périodes de hausse, de baisse et les mois pour lesquels le calcul est impossible.

In [ ]:
if donnees is not None:
    dates_graphique = donnees.index.to_timestamp()
    graphiques = [
        ("ipc", "Évolution de l’IPC au Maroc", "Indice"),
        ("variation_mensuelle_pct", "Variation mensuelle de l’IPC", "Variation (%)"),
        ("glissement_annuel_pct", "Glissement annuel de l’IPC", "Variation (%)"),
    ]

    for colonne, titre, unite in graphiques:
        if donnees[colonne].notna().any():
            fig, axe = plt.subplots(figsize=(9, 3))
            axe.plot(dates_graphique, donnees[colonne].astype(float), marker=".")
            if colonne != "ipc":
                axe.axhline(0, color="gray", linewidth=1)
            axe.set(title=titre, xlabel="Mois", ylabel=unite)
            fig.autofmt_xdate()
            fig.tight_layout()
            plt.show()
        else:
            print(f"{titre} : historique insuffisant pour calculer un taux.")

## 5. Suite du projet

À revenu nominal constant, la hausse de l’IPC réduit la quantité que ce revenu permet d’acheter au prix du panier de référence. Pour mesurer l’évolution du revenu réel en prix de l’année de base, utiliser `revenu nominal × 100 / IPC` avec des données de revenu comparables.

Pour la simulation, définir des paniers et leurs pondérations, puis comparer leur coût pour un budget donné. Préciser quelles valeurs viennent des données officielles et lesquelles relèvent des hypothèses. Pour étudier les substitutions observées, il faudra aussi des données de dépenses ou de quantités par produit.